In [ ]:
!pip install sktime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.6/37.6 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 7.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np

from xgboost import XGBRegressor
import lightgbm as lgb

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

from sklearn.metrics import r2_score

from model_classes import W5_Regs

In [ ]:
# Preset file

filename = "CRMLS_0625-0626_enriched.csv"
end_mnth = 6

In [ ]:
def load_df(file=filename):
  """
  Takes a .csv filename or filepath
  Returns the DataFrame loaded from the csv
  """
  df = pd.read_csv(file, low_memory=False)
  df["CloseDate"] = pd.to_datetime(df["CloseDate"])     # Converts "CloseDate" values to datetime type
  return df

In [ ]:
enr_df = load_df()

In [ ]:
# Preset Values

main_cols = ["BedroomsTotal", "BathroomsTotalInteger", "LivingArea", "LotSizeSquareFeet",
             "DaysOnMarket", "YearBuilt", "PostalCode", "SaleMonth"]

extras = ["ViewYN", "FireplaceYN", "NewConstructionYN", "PoolPrivateYN"]
extra_cols = [a+"_True" for a in extras] + [a+"_False" for a in extras]

totals = main_cols+extra_cols


cols = enr_df.columns.to_list()
new_col_strt = [i for i in range(len(cols)) if cols[i] == "SaleMonth"]
new_cols = cols[(new_col_strt[0]+1):len(cols)]
new_cols = [i for i in new_cols if i != "DistrictName"]

crit_cols = totals+new_cols
log_cols = main_cols+new_cols


target = "ClosePrice"

In [ ]:
class W7_Boosting():

  def __init__(self,df):
    self.df = df


  def test_train_split(self, end=end_mnth):
    """
    Takes a DataFrame
    Encodes "PropertyType" column
    Returns a defined training and test set for the DataFrame
    """
    yr_mo = []
    for i in self.df['CloseDate']:                                         # For each date in the "CloseDate" column
      yr, mo = i.year, i.month                                          # Define the year and month values of date i
      yr_mo.append([yr,mo])                                             # Append to "yr_mo" a list of date i's year and month
    te_set = [b for b in range(len(yr_mo)) if yr_mo[b] == [2026,end]]   # Define a list of row #s with date 06/2026
    te_rng = te_set[0::len(te_set)-1]                                 # Define a list of the first and last row in "te_set"
    tr, te = self.df[0:te_rng[0]], self.df[te_rng[0]:te_rng[1]]             # Define the training and test sets of the inputted df
    return tr, te


  def XGB(self, tr, te, feat=crit_cols, targ=target, dep=6, l_rate=0.3, est=150):
    """
    Takes a training DataFrame and test DataFrame
    """
    x_tr = tr[feat].values            # Define the x_train set values
    y_tr = tr[targ].values            # Define the y_train set values
    x_te = te[feat].values            # Define the x_test set values
    y_te = te[targ].values            # Define the y_test set values

    model = XGBRegressor(random_state=0, max_depth=dep,
                         learning_rate=l_rate, n_estimators=est)        # Define Decision Tree Regression model
    model.fit(x_tr, y_tr)             # Fit training data to model
    y_pred = model.predict(x_te)      # Models the predicted y values from x_test values

    return y_te, y_pred


  def GBR(self, tr, te, feat=crit_cols, targ=target, dep=3, l_rate=0.1, est=100):
    """
    Takes a training DataFrame and test DataFrame
    """
    x_tr = tr[feat].values            # Define the x_train set values
    y_tr = tr[targ].values            # Define the y_train set values
    x_te = te[feat].values            # Define the x_test set values
    y_te = te[targ].values            # Define the y_test set values

    model = GradientBoostingRegressor(random_state=0, max_depth=dep,
                                      learning_rate=l_rate, n_estimators=est)        # Define Decision Tree Regression model
    model.fit(x_tr, y_tr)             # Fit training data to model
    y_pred = model.predict(x_te)      # Models the predicted y values from x_test values

    return y_te, y_pred


  def L_GBM(self, tr, te, feat=crit_cols, targ=target, dep=-1, l_rate=0.1, est=100):
    """
    Takes a training DataFrame and test DataFrame
    """
    x_tr, y_tr = tr[feat].values, tr[targ].values       # Define the x_train, y_train set values
    x_te, y_te = te[feat].values, te[targ].values       # Define the x_test, y_test set values

    lgb_tr = lgb.Dataset(x_tr, y_tr, free_raw_data=False)
    lgb_te = lgb.Dataset(x_te, y_te, free_raw_data=False)

    model = lgb.train(params={"max_depth": dep, "learning_rate": l_rate, "n_estimators": est},
                      train_set=lgb_tr, valid_sets=lgb_tr)          # Define Decision Tree Regression model
    y_pred = model.predict(x_te, num_iteration=model.best_iteration)      # Models the predicted y values from x_test values

    return y_te, y_pred


  def HGBR(self, tr, te, feat=crit_cols, targ=target, dep=None, l_rate=0.1):
    """
    Takes a training DataFrame and test DataFrame
    """
    x_tr = tr[feat].values            # Define the x_train set values
    y_tr = tr[targ].values            # Define the y_train set values
    x_te = te[feat].values            # Define the x_test set values
    y_te = te[targ].values            # Define the y_test set values

    model = HistGradientBoostingRegressor(random_state=0, max_depth=dep, learning_rate=l_rate)             # Define Decision Tree Regression model
    model.fit(x_tr, y_tr)             # Fit training data to model
    y_pred = model.predict(x_te)      # Models the predicted y values from x_test values

    return y_te, y_pred


  def log_transform(self, feat=log_cols, targ=target):
    log_df = self.df.copy()
    for col in feat+[targ]:
      new_col = []
      for val in log_df[col]:
        new_col.append(float(np.log(val)))
      log_df[col] = pd.DataFrame(new_col)
    return log_df


  def r2_eval(self, y_test, y_pred):
    r2 = r2_score(y_test, y_pred)       # Computes the r2 score of y_test and the predicted y
    return r2

In [ ]:
Week7 = W7_Boosting(enr_df)
train, test = Week7.test_train_split()

In [ ]:
comp = W5_Regs(enr_df)

# **Gradient Boosting**



**XGBoost**

In [ ]:
if __name__ == "__main__":
  comp.lng_main(Week7.XGB, train, test, Week7.r2_eval, crit_cols)

Correlation of ['BedroomsTotal/ClosePrice']: 		 0.7671
Correlation of ['BedroomsTotal/ClosePrice', 'BathroomsTotalInteger/ClosePrice']: 		 0.8187
Correlation of ['BedroomsTotal/ClosePrice', 'BathroomsTotalInteger/ClosePrice', 'LivingArea']: 		 0.9014
Correlation of ['BedroomsTotal/ClosePrice', 'BathroomsTotalInteger/ClosePrice', 'LivingArea', 'AvgAreaCost']: 		 0.8519
Correlation of ['BedroomsTotal/ClosePrice', 'BathroomsTotalInteger/ClosePrice', 'LivingArea', 'AvgAreaCost', 'BathroomsTotalInteger']: 		 0.9376
Correlation of ['BedroomsTotal/ClosePrice', 'BathroomsTotalInteger/ClosePrice', 'LivingArea', 'AvgAreaCost', 'BathroomsTotalInteger', 'AvgLivingArea']: 		 0.9214
Correlation of ['BedroomsTotal/ClosePrice', 'BathroomsTotalInteger/ClosePrice', 'LivingArea', 'AvgAreaCost', 'BathroomsTotalInteger', 'AvgLivingArea', 'DistrictID']: 		 0.9175
Correlation of ['BedroomsTotal/ClosePrice', 'BathroomsTotalInteger/ClosePrice', 'LivingArea', 'AvgAreaCost', 'BathroomsTotalInteger', 'AvgLivingAr

In [ ]:
try_cols = ['BedroomsTotal/ClosePrice', 'BathroomsTotalInteger/ClosePrice', 'LivingArea', 'BathroomsTotalInteger', 'BathroomsTotalInteger/BedroomsTotal', 'AvgAreaLot', 'ClosePrice/DaysOnMarket', 'LotSizeSquareFeet', 'PoolPrivateYN_False', 'LivingArea/LotSizeSquareFeet', 'LivingArea/DaysOnMarket', 'YearBuilt', 'SaleMonth', 'NewConstructionYN_False']

y_te, y_pr = Week7.XGB(train, test, try_cols)
r2 = Week7.r2_eval(y_te, y_pr)
print(f"{round(r2,4)}")

***Log Transform DataFrame***

In [ ]:
log_df = Week7.log_transform()
Wk7 = W7_Boosting(log_df)
tr, te = Wk7.test_train_split()

In [ ]:
if __name__ == "__main__":
  comp.shrt_main(Wk7.XGB, tr, te, Wk7.r2_eval, crit_cols)

Correlation of ['BedroomsTotal/ClosePrice', 'AvgAreaCost', 'BathroomsTotalInteger/ClosePrice', 'DistrictID', 'PostalCode', 'AvgLivingArea', 'LivingArea', 'AvgAreaLot', 'BathroomsTotalInteger', 'ClosePrice/DaysOnMarket', 'BathroomsTotalInteger/BedroomsTotal', 'BedroomsTotal', 'LotSizeSquareFeet', 'LivingArea/DaysOnMarket', 'PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True', 'LivingArea/LotSizeSquareFeet', 'PoolPrivateYN_True', 'LotSizeSquareFeet/DaysOnMarket', 'YearBuilt', 'DaysOnMarket', 'ViewYN_False']:
 0.9989


**Results**


***Non-Transform***

Highest R2 Score:  **0.9591**

---

***Log Transform***

Highest R2 Score:  **0.9989**

**Gradient Boosting Regressor**

In [ ]:
if __name__ == "__main__":
  comp.lng_main(Week7.GBR, train, test, Week7.r2_eval, crit_cols)

Correlation of ['BedroomsTotal/ClosePrice']: 		 0.8631
Correlation of ['BedroomsTotal/ClosePrice', 'BathroomsTotalInteger/ClosePrice']: 		 0.8656
Correlation of ['BedroomsTotal/ClosePrice', 'BathroomsTotalInteger/ClosePrice', 'LivingArea']: 		 0.9342
Correlation of ['BedroomsTotal/ClosePrice', 'BathroomsTotalInteger/ClosePrice', 'LivingArea', 'AvgAreaCost']: 		 0.9483
Correlation of ['BedroomsTotal/ClosePrice', 'BathroomsTotalInteger/ClosePrice', 'LivingArea', 'AvgAreaCost', 'BathroomsTotalInteger']: 		 0.9754
Correlation of ['BedroomsTotal/ClosePrice', 'BathroomsTotalInteger/ClosePrice', 'LivingArea', 'AvgAreaCost', 'BathroomsTotalInteger', 'AvgLivingArea']: 		 0.9759
Correlation of ['BedroomsTotal/ClosePrice', 'BathroomsTotalInteger/ClosePrice', 'LivingArea', 'AvgAreaCost', 'BathroomsTotalInteger', 'AvgLivingArea', 'PostalCode']: 		 0.975
Correlation of ['BedroomsTotal/ClosePrice', 'BathroomsTotalInteger/ClosePrice', 'LivingArea', 'AvgAreaCost', 'BathroomsTotalInteger', 'AvgLivingAre

***Log Transform DataFrame***

In [ ]:
if __name__ == "__main__":
  comp.shrt_main(Wk7.GBR, tr, te, Wk7.r2_eval, crit_cols)

Correlation of ['BedroomsTotal/ClosePrice', 'AvgAreaCost', 'BathroomsTotalInteger/ClosePrice', 'PostalCode', 'DistrictID', 'AvgLivingArea', 'LivingArea', 'BathroomsTotalInteger', 'AvgAreaLot', 'ClosePrice/DaysOnMarket', 'BathroomsTotalInteger/BedroomsTotal', 'BedroomsTotal']:
 0.9976


**Results**


***Non-Transform***

Highest R2 Score:  **0.9931**

---

***Log Transform***

Highest R2 Score:  **0.9976**

---

* Less difference before Transform and Non-Transform R2 scores than XGBoost

**LightGBM**

In [ ]:
if __name__ == "__main__":
  comp.shrt_main(Week7.L_GBM, train, test, Week7.r2_eval, crit_cols)

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

***Log Transform DataFrame***

In [ ]:
if __name__ == "__main__":
  comp.shrt_main(Wk7.L_GBM, tr, te, Wk7.r2_eval, crit_cols)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001550 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 14
[LightGBM] [Info] Number of data points in the train set: 92986, number of used features: 1
[LightGBM] [Info] Start training from score 13.785278
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

**Results**


***Non-Transform***

Highest R2 Score:  **0.9502**

---

***Log Transform***

Highest R2 Score:  **0.999**

---

 * Results are similar to XGBoost

**Histogram-based Gradient Boosting Regressor**

In [ ]:
if __name__ == "__main__":
  comp.lng_main(Week7.HGBR, train, test, Week7.r2_eval, crit_cols)

Correlation of ['BedroomsTotal/ClosePrice']: 		 0.7634
Correlation of ['BedroomsTotal/ClosePrice', 'BathroomsTotalInteger/ClosePrice']: 		 0.81
Correlation of ['BedroomsTotal/ClosePrice', 'BathroomsTotalInteger/ClosePrice', 'LivingArea']: 		 0.9073
Correlation of ['BedroomsTotal/ClosePrice', 'BathroomsTotalInteger/ClosePrice', 'LivingArea', 'AvgAreaCost']: 		 0.907
Correlation of ['BedroomsTotal/ClosePrice', 'BathroomsTotalInteger/ClosePrice', 'LivingArea', 'AvgAreaCost', 'BathroomsTotalInteger']: 		 0.9398
Correlation of ['BedroomsTotal/ClosePrice', 'BathroomsTotalInteger/ClosePrice', 'LivingArea', 'AvgAreaCost', 'BathroomsTotalInteger', 'AvgLivingArea']: 		 0.9338
Correlation of ['BedroomsTotal/ClosePrice', 'BathroomsTotalInteger/ClosePrice', 'LivingArea', 'AvgAreaCost', 'BathroomsTotalInteger', 'AvgLivingArea', 'BathroomsTotalInteger/BedroomsTotal']: 		 0.9332
Correlation of ['BedroomsTotal/ClosePrice', 'BathroomsTotalInteger/ClosePrice', 'LivingArea', 'AvgAreaCost', 'BathroomsTotal

In [ ]:
try_cols = ['BedroomsTotal/ClosePrice', 'BathroomsTotalInteger/ClosePrice', 'LivingArea', 'BathroomsTotalInteger', 'DistrictID', 'AvgAreaLot', 'BedroomsTotal', 'ClosePrice/DaysOnMarket', 'LotSizeSquareFeet', 'FireplaceYN_False', 'LivingArea/DaysOnMarket', 'LivingArea/LotSizeSquareFeet', 'ViewYN_True', 'ViewYN_False', 'SaleMonth', 'DaysOnMarket', 'NewConstructionYN_True']

y_te, y_pr = Week7.HGBR(train, test, try_cols)
r2 = Week7.r2_eval(y_te, y_pr)
print(f"{round(r2,4)}")

***Log Transform DataFrame***

In [ ]:
if __name__ == "__main__":
  comp.shrt_main(Wk7.HGBR, tr, te, Wk7.r2_eval, crit_cols)

Correlation of ['BedroomsTotal/ClosePrice', 'AvgAreaCost', 'BathroomsTotalInteger/ClosePrice', 'PostalCode', 'DistrictID', 'AvgLivingArea', 'LivingArea', 'AvgAreaLot', 'BathroomsTotalInteger', 'ClosePrice/DaysOnMarket', 'BathroomsTotalInteger/BedroomsTotal', 'BedroomsTotal', 'LotSizeSquareFeet', 'LivingArea/DaysOnMarket', 'PoolPrivateYN_False', 'FireplaceYN_False']:
 0.9987


**Results**


***Non-Transform***

Highest R2 Score:  **0.9536**

---

***Log Transform***

Highest R2 Score:  **0.9987**

---

* Results are similar to XGBoost and LightGBM

# **Hyperparameter Tuning**

***Note:*** These model iterations had to be tested individually due to the processing time.

***Note:*** Also, all models can be run as they are *OR, to reduce processing time,* **the log can be run before the non-transform of each model, and the best values of depth,lr,n_est (commented out in the model code) for the log model may be inputted into the non-transform of the model. For loop values with comment "# Note" should then be commented out and the proceeding values unindented.**

 * Example:
   * Run 'XGBoost - log'
   * Go to 'XGBoost'
   * Uncomment row with "depth,lr,n_est = "
   * Comment out rows with comment "# Note"
   * Unindent preceeding rows in for loop

In [ ]:
r2_dict = {"Model": [], "LogForm": [], "ScoreType": [], "ScoreValue": [], "max_depth": [], "learning_rate": [], "n_estimators": [], "columns": []}

In [ ]:
# XGBoost

r2_cols = comp.shrt_main(Week7.XGB, train, test, Week7.r2_eval, crit_cols, prt=False)

r2_scrs = []
# depth,lr,n_est = r2_dict["max_depth"][0], r2_dict["learning_rate"][0], r2_dict["n_estimators"][0]         # Uses best values from log of model
for depth in range(2,11):           # Note
  for learn_rate in range(1,6):     # Note
    lr = learn_rate/10              # Note
    for n_est in range(50,201,50):  # Note
      y_te, y_pr = Week7.XGB(train, test, feat=r2_cols[0], dep=depth, l_rate=lr, est=n_est)
      r2 = Week7.r2_eval(y_te, y_pr)
      all = [round(r2,4), depth, lr, n_est, r2_cols[0]]
      r2_scrs.append(all)
r2_scrs = sorted(r2_scrs, key=lambda x: x[0], reverse=True)
r2_scrs[0].insert(0, "R2 Score")
r2_scrs[0].insert(0, False)
r2_scrs[0].insert(0, "XGBoost")
for i in range(len(r2_scrs[0])):
  r2_dict[list(r2_dict.keys())[i]].append(r2_scrs[0][i])

In [ ]:
# XGBoost - log

r2_cols = comp.shrt_main(Wk7.XGB, tr, te, Wk7.r2_eval, crit_cols, prt=False)

r2_scrs = []
for depth in range(2,11):
  for learn_rate in range(1,6):
    lr = learn_rate/10
    for n_est in range(50,201,50):
      y_te, y_pr = Wk7.XGB(tr, te, feat=r2_cols[0], dep=depth, l_rate=lr, est=n_est)
      r2 = Wk7.r2_eval(y_te, y_pr)
      all = [round(r2,4), depth, lr, n_est, r2_cols[0]]
      r2_scrs.append(all)
r2_scrs = sorted(r2_scrs, key=lambda x: x[0], reverse=True)
r2_scrs[0].insert(0, "R2 Score")
r2_scrs[0].insert(0, True)
r2_scrs[0].insert(0, "XGBoost")
for i in range(len(r2_scrs[0])):
  r2_dict[list(r2_dict.keys())[i]].append(r2_scrs[0][i])

Correlation of ['BedroomsTotal/ClosePrice', 'AvgAreaCost', 'BathroomsTotalInteger/ClosePrice', 'DistrictID', 'PostalCode', 'AvgLivingArea', 'LivingArea', 'AvgAreaLot', 'BathroomsTotalInteger', 'ClosePrice/DaysOnMarket', 'BathroomsTotalInteger/BedroomsTotal', 'BedroomsTotal', 'LotSizeSquareFeet', 'LivingArea/DaysOnMarket', 'PoolPrivateYN_False', 'FireplaceYN_False', 'FireplaceYN_True', 'LivingArea/LotSizeSquareFeet', 'PoolPrivateYN_True', 'LotSizeSquareFeet/DaysOnMarket', 'YearBuilt', 'DaysOnMarket', 'ViewYN_False']:
 0.9989


In [ ]:
# Gradient Boosting Regressor

r2_cols =   comp.shrt_main(Week7.GBR, train, test, Week7.r2_eval, crit_cols, prt=False)

r2_scrs = []
# depth,lr,n_est = r2_dict["max_depth"][2], r2_dict["learning_rate"][2], r2_dict["n_estimators"][2]         # Uses best values from log of model
for depth in range(1,7):
  for learn_rate in range(25,351,25):
    lr = learn_rate/100
    for n_est in range(50,201,50):
      y_te, y_pr = Week7.GBR(train, test, feat=r2_cols[0], dep=depth, l_rate=lr, est=n_est)
      r2 = Week7.r2_eval(y_te, y_pr)
      all = [round(r2,4), depth, lr, n_est, r2_cols[0]]
      r2_scrs.append(all)
r2_scrs = sorted(r2_scrs, key=lambda x: x[0], reverse=True)
r2_scrs[0].insert(0, "R2 Score")
r2_scrs[0].insert(0, False)
r2_scrs[0].insert(0, "Gradient Boosting Regressor")
for i in range(len(r2_scrs[0])):
  r2_dict[list(r2_dict.keys())[i]].append(r2_scrs[0][i])

In [ ]:
# Gradient Boosting Regressor - log

r2_cols = comp.shrt_main(Wk7.GBR, tr, te, Wk7.r2_eval, crit_cols, prt=False)

r2_scrs = []
for depth in range(1,7):
  for learn_rate in range(25,351,25):
    lr = learn_rate/100
    for n_est in range(50,201,50):
      y_te, y_pr = Wk7.GBR(tr, te, feat=r2_cols[0], dep=depth, l_rate=lr, est=n_est)
      r2 = Wk7.r2_eval(y_te, y_pr)
      all = [round(r2,4), depth, lr, n_est, r2_cols[0]]
      r2_scrs.append(all)
r2_scrs = sorted(r2_scrs, key=lambda x: x[0], reverse=True)
r2_scrs[0].insert(0, "R2 Score")
r2_scrs[0].insert(0, True)
r2_scrs[0].insert(0, "Gradient Boosting Regressor")
for i in range(len(r2_scrs[0])):
  r2_dict[list(r2_dict.keys())[i]].append(r2_scrs[0][i])

In [ ]:
# LightGBM

r2_cols = comp.shrt_main(Week7.L_GBM, train, test, Week7.r2_eval, crit_cols, prt=False)

r2_scrs = []
# depth,lr,n_est = r2_dict["max_depth"][4], r2_dict["learning_rate"][4], r2_dict["n_estimators"][4]         # Uses best values from log of model
for depth in range(1,7):
  for learn_rate in range(25,351,25):
    lr = learn_rate/100
    for n_est in range(50,201,50):
      y_te, y_pr = Week7.L_GBM(train, test, feat=r2_cols[0], dep=depth, l_rate=lr, est=n_est)
      r2 = Week7.r2_eval(y_te, y_pr)
      all = [round(r2,4), depth, lr, n_est, r2_cols[0]]
      r2_scrs.append(all)
r2_scrs = sorted(r2_scrs, key=lambda x: x[0], reverse=True)
r2_scrs[0].insert(0, "R2 Score")
r2_scrs[0].insert(0, False)
r2_scrs[0].insert(0, "LightGBM")
for i in range(len(r2_scrs[0])):
  r2_dict[list(r2_dict.keys())[i]].append(r2_scrs[0][i])

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002303 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15
[LightGBM] [Info] Number of data points in the train set: 92986, number of used features: 1
[LightGBM] [Info] Start training from score 1254042.147315
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

In [ ]:
# LightGBM - log

r2_cols = comp.shrt_main(Wk7.L_GBM, tr, te, Wk7.r2_eval, crit_cols, prt=False)

r2_scrs = []
for depth in range(1,7):
  for learn_rate in range(25,351,25):
    lr = learn_rate/100
    for n_est in range(50,201,50):
      y_te, y_pr = Wk7.L_GBM(tr, te, feat=r2_cols[0], dep=depth, l_rate=lr, est=n_est)
      r2 = Wk7.r2_eval(y_te, y_pr)
      all = [round(r2,4), depth, lr, n_est, r2_cols[0]]
      r2_scrs.append(all)
r2_scrs = sorted(r2_scrs, key=lambda x: x[0], reverse=True)
r2_scrs[0].insert(0, "R2 Score")
r2_scrs[0].insert(0, True)
r2_scrs[0].insert(0, "LightGBM")
for i in range(len(r2_scrs[0])):
  r2_dict[list(r2_dict.keys())[i]].append(r2_scrs[0][i])

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

In [ ]:
# Histogram-based Gradient Boosting Regressor

r2_cols = comp.shrt_main(Week7.HGBR, train, test, Week7.r2_eval, crit_cols, prt=False)

r2_scrs = []
# depth,lr,n_est = r2_dict["max_depth"][6], r2_dict["learning_rate"][6], r2_dict["n_estimators"][6]         # Uses best values from log of model
for depth in range(1,7):
  for learn_rate in range(25,351,25):
    lr = learn_rate/100
    y_te, y_pr = Week7.HGBR(train, test, feat=r2_cols[0], dep=depth, l_rate=lr)
    r2 = Week7.r2_eval(y_te, y_pr)
    all = [round(r2,4), depth, lr, None, r2_cols[0]]
    r2_scrs.append(all)
r2_scrs = sorted(r2_scrs, key=lambda x: x[0], reverse=True)
r2_scrs[0].insert(0, "R2 Score")
r2_scrs[0].insert(0, False)
r2_scrs[0].insert(0, "Histogram-based Gradient Boosting Regressor")
for i in range(len(r2_scrs[0])):
  r2_dict[list(r2_dict.keys())[i]].append(r2_scrs[0][i])

In [ ]:
# Histogram-based Gradient Boosting Regressor - log

r2_cols = comp.shrt_main(Wk7.HGBR, tr, te, Wk7.r2_eval, crit_cols, prt=False)

r2_scrs = []
for depth in range(1,7):
  for learn_rate in range(25,351,25):
    lr = learn_rate/100
    y_te, y_pr = Wk7.HGBR(tr, te, feat=r2_cols[0], dep=depth, l_rate=lr)
    r2 = Wk7.r2_eval(y_te, y_pr)
    all = [round(r2,4), depth, lr, None, r2_cols[0]]
    r2_scrs.append(all)
r2_scrs = sorted(r2_scrs, key=lambda x: x[0], reverse=True)
r2_scrs[0].insert(0, "R2 Score")
r2_scrs[0].insert(0, True)
r2_scrs[0].insert(0, "Histogram-based Gradient Boosting Regressor")
for i in range(len(r2_scrs[0])):
  r2_dict[list(r2_dict.keys())[i]].append(r2_scrs[0][i])

***Best Feature Values:***

 * *XGBoost*
   * Non-Transform
     * max_depth = **7**, learning_rate = **0.2**, n_estimators = **200**
   * Log Transform
     * max_depth = **6**, learning_rate = **0.3**, n_estimators = **200**
 * *GBR*
   * Non-Transform
     * max_depth = **3**, learning_rate = **0.25**, n_estimators = **200**
   * Log Transform
     * max_depth = **6**, learning_rate = **0.25**, n_estimators = **150**
 * *LightGBM*
   * Non-Transform
     * max_depth = **5**, learning_rate = **0.5**, n_estimators = **200**
   * Log Transform
     * max_depth = **5**, learning_rate = **0.5**, n_estimators = **200**
 * *HGBR*
   * Non-Transform
     * max_depth = **5**, learning_rate = **0.75**
   * Log Transform
     * max_depth = **6**, learning_rate = **0.25**

In [ ]:
def main():
  result_df = pd.DataFrame(r2_dict)
  return result_df

In [ ]:
results = main()
results

,Model,LogForm,ScoreType,ScoreValue,max_depth,learning_rate,n_estimators,columns
0,XGBoost,True,R2 Score,0.9990,6,0.30,200.0,"[BedroomsTotal/ClosePrice, AvgAreaCost, Bathro..."
1,XGBoost,False,R2 Score,0.9572,6,0.30,200.0,"[BedroomsTotal/ClosePrice, BathroomsTotalInteg..."
2,Gradient Boosting Regressor,True,R2 Score,0.9998,6,0.25,150.0,"[BedroomsTotal/ClosePrice, AvgAreaCost, Bathro..."
3,Gradient Boosting Regressor,False,R2 Score,0.9870,6,0.25,150.0,"[BedroomsTotal/ClosePrice, BathroomsTotalInteg..."
4,LightGBM,True,R2 Score,0.9990,5,0.50,200.0,"[BedroomsTotal/ClosePrice, AvgAreaCost, Bathro..."
5,LightGBM,False,R2 Score,0.9604,5,0.50,200.0,"[BedroomsTotal/ClosePrice, BathroomsTotalInteg..."
6,Histogram-based Gradient Boosting Regressor,True,R2 Score,0.9986,6,0.25,NaN,"[BedroomsTotal/ClosePrice, AvgAreaCost, Bathro..."
7,Histogram-based Gradient Boosting Regressor,False,R2 Score,0.9549,6,0.25,NaN,"[BedroomsTotal/ClosePrice, BathroomsTotalInteg..."


In [ ]:
def save_csv(df, file):
  """
  Takes a DataFrame
  Saves the inputted DataFrame as a .csv file, given inputted name
  """
  df.to_csv(file, index=False)

In [ ]:
save_csv(results, "boosting_r2s.csv")